# 01 Attractions: locations and check map

This notebook loads the hand-made attractions table (`data/attractions.csv`), adds coordinates and draws a check map.

**Roles:** `calibration` = paid sites with ticket counts, used to fit the model; `competitor` = in the choice set for the Huff model; `separate class` = different visit motive, probably excluded; `context` = free sites without counts, shown on maps only.

**Geocoding:** coordinates come from OpenStreetMap via Nominatim (© OpenStreetMap contributors), using Russian search queries. Results are cached in `data/processed/attractions_coords.csv`, so each place is queried only once. Places Nominatim cannot find, or finds in the wrong spot, get hand coordinates in `MANUAL`.

In [1]:
import os
import pandas as pd
import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [2]:
# Load the attractions table
att = pd.read_csv("../data/attractions.csv")
print(att.shape)
print(att["role"].value_counts())
att[["name_en", "district", "type", "role", "annual_visitors"]]

(13, 11)
role
calibration    9
context        3
competitor     1
Name: count, dtype: int64


,name_en,district,type,role,annual_visitors
0,Slantsy Local History Museum,Slantsevsky,museum,calibration,10000.0
1,Kingisepp Local History Museum,Kingiseppsky,museum,calibration,11000.0
2,Ivangorod Museum (fortress and galleries),Kingiseppsky,fortress / museum,calibration,NaN
3,Koporye Fortress,Lomonosovsky,fortress,calibration,NaN
4,Staraya Ladoga Museum-Reserve,Volkhovsky,museum-reserve,calibration,140000.0
5,Sablino caves and waterfalls,Tosnensky,natural site / caves,calibration,NaN
6,Putilovo Paleopark (fossil digging),Kirovsky,geology park,calibration,NaN
7,Gdov Kremlin and Local Museum,Gdovsky (Pskov Oblast),fortress / museum,competitor,NaN
8,Oreshek Fortress (Shlisselburg),Kirovsky,fortress,calibration,NaN
9,Roerich Estate Museum (Izvara),Volosovsky,museum,calibration,NaN


In [3]:
# Russian search queries for Nominatim (one per attraction)
QUERIES = {
    "Slantsy Local History Museum": "Сланцы, улица Кирова, 14",
    "Kingisepp Local History Museum": "Кингисепп, проспект Карла Маркса, 1",
    "Ivangorod Museum (fortress and galleries)": "Ивангородская крепость",
    "Koporye Fortress": "Копорская крепость",
    "Staraya Ladoga Museum-Reserve": "Староладожская крепость",
    "Sablino caves and waterfalls": "Саблинские пещеры",
    "Putilovo Paleopark (fossil digging)": "Путилово, Кировский район",
    "Gdov Kremlin and Local Museum": "Гдовский кремль",
    "Oreshek Fortress (Shlisselburg)": "крепость Орешек",
    "Roerich Estate Museum (Izvara)": "Извара, Волосовский район",
    "Yam Fortress ramparts (Kingisepp)": "крепость Ям, Кингисепп",
    "Lopukhinka radon springs": "Лопухинка, Ломоносовский район",
    "Lava river canyon": "Городище, Кировский район, Ленинградская область",
}

# Hand coordinates override geocoding: name_en -> (lat, lon)
MANUAL = {
    "Putilovo Paleopark (fossil digging)": (59.856218, 31.401089),  # kp.ru article
    "Slantsy Local History Museum": (59.125272, 28.089061),  # Yandex Maps
    "Sablino caves and waterfalls": (59.666998, 30.796894),  # Yandex Maps, cave entrance
}

# Cache holds only successful lookups, so failed ones are retried on the next run
CACHE = "../data/processed/attractions_coords.csv"
if os.path.exists(CACHE):
    coords = pd.read_csv(CACHE)
else:
    coords = pd.DataFrame(columns=["name_en", "query", "lat", "lon", "found_name"])

geolocator = Nominatim(user_agent="klint-visitor-demand-model", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1,
                      max_retries=2, error_wait_seconds=5)

new_rows, failed = [], []
for name, query in QUERIES.items():
    if name in coords["name_en"].values or name in MANUAL:
        continue
    loc = geocode(query, language="ru", country_codes="ru")
    if loc:
        new_rows.append({"name_en": name, "query": query,
                         "lat": loc.latitude, "lon": loc.longitude,
                         "found_name": loc.address})
    else:
        failed.append(name)

if new_rows:
    coords = pd.concat([coords, pd.DataFrame(new_rows)], ignore_index=True)
    os.makedirs("../data/processed", exist_ok=True)
    coords.to_csv(CACHE, index=False)

print("Not found or connection failed:", failed)

# Add manual coordinates (they replace any geocoded result)
manual = pd.DataFrame(
    [{"name_en": n, "lat": lat, "lon": lon, "found_name": "manual"}
     for n, (lat, lon) in MANUAL.items()]
)
all_coords = pd.concat(
    [coords[~coords["name_en"].isin(MANUAL)], manual], ignore_index=True
)

att = pd.read_csv("../data/attractions.csv")  # reload so re-running doesn't duplicate columns
att = att.merge(all_coords[["name_en", "lat", "lon", "found_name"]], on="name_en", how="left")
print("Missing coordinates:", att["lat"].isna().sum())
att[["name_en", "lat", "lon", "found_name"]]

Not found or connection failed: []
Missing coordinates: 0


,name_en,lat,lon,found_name
0,Slantsy Local History Museum,59.125272,28.089061,manual
1,Kingisepp Local History Museum,59.373503,28.601264,"7/1, проспект Карла Маркса, Кингисепп, Кингисе..."
2,Ivangorod Museum (fortress and galleries),59.374551,28.206757,"Ивангородская крепость, Девичья гора, Ивангоро..."
3,Koporye Fortress,59.709207,29.031764,"Копорская крепость, 41К-014, Копорье, Копорско..."
4,Staraya Ladoga Museum-Reserve,59.997691,32.298198,"Староладожская крепость, Варяжская улица, Стар..."
5,Sablino caves and waterfalls,59.666998,30.796894,manual
6,Putilovo Paleopark (fossil digging),59.856218,31.401089,manual
7,Gdov Kremlin and Local Museum,58.740543,27.819774,"Гдовский кремль, Поцелуев мост, Гдов, Гдовский..."
8,Oreshek Fortress (Shlisselburg),59.954165,31.038763,"крепость Орешек, улица Новоладожский канал, Пр..."
9,Roerich Estate Museum (Izvara),59.353513,29.519527,"Извара, Рабитицкое сельское поселение, Волосов..."


In [4]:
# Check map, coloured by role
ROLE_COLORS = {
    "calibration": "#1b3a6b",
    "competitor": "#ef8a62",
    "separate class": "#9e7bb5",
    "context": "#999999",
}

m = folium.Map(
    location=[59.6, 30.5], zoom_start=8,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
)
for _, r in att.dropna(subset=["lat"]).iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]], radius=7,
        color=ROLE_COLORS[r["role"]], fill=True, fill_opacity=0.9,
        tooltip=f"{r['name_en']} ({r['role']})",
    ).add_to(m)
m